# 02 — Baseline Models
Popularity baseline + Matrix Factorization (SVD for explicit, ALS for implicit).
**Requires:** `01_data_preprocessing.ipynb` to have been run first.

In [1]:
import pandas as pd
import numpy as np
import pickle, itertools, time
from pathlib import Path
from tqdm.auto import tqdm

ROOT = Path('..').resolve()
PROC = ROOT / 'data' / 'processed'

def load_dataset(name):
    p = PROC / name
    train = pd.read_parquet(p / 'train.parquet')
    val   = pd.read_parquet(p / 'val.parquet')
    test  = pd.read_parquet(p / 'test.parquet')
    meta  = pickle.load(open(p / 'meta.pkl', 'rb'))
    print(f'Loaded {name}: n_users={meta["n_users"]:,}  n_items={meta["n_items"]:,}')
    return train, val, test, meta

ml_train,     ml_val,     ml_test,     ml_meta     = load_dataset('ml-100k')
lastfm_train, lastfm_val, lastfm_test, lastfm_meta = load_dataset('lastfm-2k')

n_users_ml,     n_items_ml     = ml_meta['n_users'],     ml_meta['n_items']
n_users_lastfm, n_items_lastfm = lastfm_meta['n_users'], lastfm_meta['n_items']
all_items_ml = np.arange(n_items_ml)
all_items_lastfm = np.arange(n_items_lastfm)

Loaded ml-100k: n_users=938  n_items=1,008
Loaded lastfm-2k: n_users=1,859  n_items=2,823


## Evaluation metrics

In [2]:
def hit_rate_at_k(recs, gt, k=10):
    hits, total = 0, 0
    for u, r in recs.items():
        g = set(gt.get(u, []))
        if not g: continue
        hits += int(len(set(r[:k]) & g) > 0)
        total += 1
    return hits / total if total else 0.0

def ndcg_at_k(recs, gt, k=10):
    scores = []
    for u, r in recs.items():
        g = set(gt.get(u, []))
        if not g: continue
        dcg  = sum(1/np.log2(i+2) for i,it in enumerate(r[:k]) if it in g)
        idcg = sum(1/np.log2(i+2) for i in range(min(len(g), k)))
        scores.append(dcg/idcg if idcg else 0.0)
    return float(np.mean(scores)) if scores else 0.0

def map_at_k(recs, gt, k=10):
    aps = []
    for u, r in recs.items():
        g = set(gt.get(u, []))
        if not g: continue
        hits, ps = 0, 0.0
        for rank, it in enumerate(r[:k]):
            if it in g:
                hits += 1; ps += hits/(rank+1)
        aps.append(ps / min(len(g), k))
    return float(np.mean(aps)) if aps else 0.0

def metrics(recs, gt, k=10):
    return {f'hr@{k}': hit_rate_at_k(recs,gt,k),
            f'ndcg@{k}': ndcg_at_k(recs,gt,k),
            f'map@{k}':  map_at_k(recs,gt,k)}

def cold_start_metrics(recs, test_df, k=10):
    gt_all = test_df.groupby('user_id')['item_id'].apply(list).to_dict()
    cold    = set(test_df[test_df.cold_start==True].user_id)
    regular = set(test_df[test_df.cold_start==False].user_id)
    return {
        'cold_start': metrics({u:r for u,r in recs.items() if u in cold},
                              {u:g for u,g in gt_all.items() if u in cold}, k),
        'regular':    metrics({u:r for u,r in recs.items() if u in regular},
                              {u:g for u,g in gt_all.items() if u in regular}, k),
    }

print('Evaluation functions defined.')

Evaluation functions defined.


## Popularity baseline

In [3]:
class PopularityBaseline:
    """Recommend globally most popular unseen items to every user."""
    def fit(self, train_df, item_col='item_id'):
        self.popular_items = train_df[item_col].value_counts().index.to_numpy()
        return self
    def recommend(self, user_ids, train_df, k=10, user_col='user_id', item_col='item_id'):
        seen = train_df.groupby(user_col)[item_col].apply(set).to_dict()
        return {u: [it for it in self.popular_items if it not in seen.get(u,set())][:k]
                for u in user_ids}

# MovieLens
pop_ml = PopularityBaseline().fit(ml_train)
pop_recs_ml = pop_ml.recommend(ml_test.user_id.unique(), ml_train, k=10)
gt_ml = ml_test.groupby('user_id')['item_id'].apply(list).to_dict()
pop_m_ml = metrics(pop_recs_ml, gt_ml)
print('Popularity — MovieLens:', pop_m_ml)

# LastFM-2K
pop_lastfm = PopularityBaseline().fit(lastfm_train)
pop_recs_lastfm = pop_lastfm.recommend(lastfm_test.user_id.unique(), lastfm_train, k=10)
gt_lastfm = lastfm_test.groupby('user_id')['item_id'].apply(list).to_dict()
pop_m_lastfm = metrics(pop_recs_lastfm, gt_lastfm)
print('Popularity — LastFM-2K:', pop_m_lastfm)

# Alias Amazon Music popularity and ground truth to LastFM-2K data for this workspace
pop_m_amz = pop_m_lastfm
gt_amz = gt_lastfm

Popularity — MovieLens: {'hr@10': 0.2505353319057816, 'ndcg@10': 0.053270628576867256, 'map@10': 0.02581406361021687}
Popularity — LastFM-2K: {'hr@10': 0.3423180592991914, 'ndcg@10': 0.1023906833503484, 'map@10': 0.056819513969109656}


In [4]:
import sys
print(sys.executable)
import surprise
print(surprise.__version__)

/Users/XuanNguyen/Documents/NUS/DSS5104/dss5104_recommendation_system/.venv311/bin/python
1.1.4


## SVD — explicit feedback (MovieLens)

In [5]:
import surprise
import warnings
warnings.filterwarnings('ignore')

def fit_svd(train_df, n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02):
    reader   = surprise.Reader(rating_scale=(0,1))
    surprise_df = train_df[['user_id','item_id','label']].rename(
        columns={'user_id':'userID','item_id':'itemID','label':'rating'})
    data     = surprise.Dataset.load_from_df(surprise_df, reader)
    trainset = data.build_full_trainset()
    model    = surprise.SVD(n_factors=n_factors, n_epochs=n_epochs,
                            lr_all=lr_all, reg_all=reg_all, verbose=False)
    t0 = time.time()
    model.fit(trainset)
    print(f'  SVD fitted {time.time()-t0:.1f}s | factors={n_factors} reg={reg_all}')
    return model

def svd_recommend(model, user_ids, train_df, all_items, k=10):
    seen = train_df.groupby('user_id')['item_id'].apply(set).to_dict()
    recs = {}
    for uid in user_ids:
        candidates = [it for it in all_items if it not in seen.get(uid,set())]
        scores     = np.array([model.predict(str(uid),str(it)).est for it in candidates])
        top_k      = np.argsort(scores)[::-1][:k]
        recs[uid]  = [candidates[i] for i in top_k]
    return recs

print('SVD functions defined.')

SVD functions defined.


In [6]:
# Tune SVD on validation set
val_users_ml = ml_val.user_id.unique()
gt_val_ml    = ml_val.groupby('user_id')['item_id'].apply(list).to_dict()
svd_results  = []

for n_factors, reg_all in tqdm(list(itertools.product([50,100,200],[0.01,0.02,0.05])),
                                desc='SVD tuning'):
    m     = fit_svd(ml_train, n_factors=n_factors, reg_all=reg_all)
    recs  = svd_recommend(m, val_users_ml, ml_train, all_items_ml, k=10)
    svd_results.append({'n_factors':n_factors,'reg_all':reg_all,
                        **metrics(recs, gt_val_ml)})

svd_df = pd.DataFrame(svd_results).sort_values('ndcg@10', ascending=False)
print('SVD tuning results:')
svd_df

SVD tuning:   0%|          | 0/9 [00:00<?, ?it/s]

  SVD fitted 0.1s | factors=50 reg=0.01
  SVD fitted 0.1s | factors=50 reg=0.02
  SVD fitted 0.1s | factors=50 reg=0.05
  SVD fitted 0.1s | factors=100 reg=0.01
  SVD fitted 0.1s | factors=100 reg=0.02
  SVD fitted 0.1s | factors=100 reg=0.05
  SVD fitted 0.2s | factors=200 reg=0.01
  SVD fitted 0.2s | factors=200 reg=0.02
  SVD fitted 0.2s | factors=200 reg=0.05
SVD tuning results:


,n_factors,reg_all,hr@10,ndcg@10,map@10
0,50,0.01,0.059957,0.006916,0.002087
1,50,0.02,0.059957,0.006916,0.002087
2,50,0.05,0.059957,0.006916,0.002087
3,100,0.01,0.059957,0.006916,0.002087
4,100,0.02,0.059957,0.006916,0.002087
5,100,0.05,0.059957,0.006916,0.002087
6,200,0.01,0.059957,0.006916,0.002087
7,200,0.02,0.059957,0.006916,0.002087
8,200,0.05,0.059957,0.006916,0.002087


In [7]:
# Best SVD on test set
best = svd_df.iloc[0]
best_svd = fit_svd(ml_train, n_factors=int(best.n_factors), reg_all=float(best.reg_all))
svd_recs_ml = svd_recommend(best_svd, ml_test.user_id.unique(), ml_train, all_items_ml, k=10)
svd_m_ml    = metrics(svd_recs_ml, gt_ml)
print(f'Best SVD (factors={int(best.n_factors)}, reg={best.reg_all}) test metrics:')
print(svd_m_ml)

print('\nCold-start breakdown:')
for group, m in cold_start_metrics(svd_recs_ml, ml_test).items():
    print(f'  {group}: {m}')

  SVD fitted 0.1s | factors=50 reg=0.01
Best SVD (factors=50, reg=0.01) test metrics:
{'hr@10': 0.05353319057815846, 'ndcg@10': 0.005492850383026032, 'map@10': 0.0015152760783114102}

Cold-start breakdown:
  cold_start: {'hr@10': 0.0, 'ndcg@10': 0.0, 'map@10': 0.0}
  regular: {'hr@10': 0.05382131324004306, 'ndcg@10': 0.005522413625130587, 'map@10': 0.0015234314931570046}


## ALS — implicit feedback (Amazon Music)

In [8]:
import implicit
import scipy.sparse as sp

def build_sparse(train_df, n_users, n_items):
    counts = train_df.groupby(['user_id','item_id']).size().reset_index(name='c')
    return sp.csr_matrix((counts.c.values,
                          (counts.user_id.values, counts.item_id.values)),
                         shape=(n_users, n_items))

def fit_als(train_df, n_users, n_items,
            factors=128, iterations=20, regularization=0.01, alpha=40.0):
    user_item = build_sparse(train_df, n_users, n_items)
    model = implicit.als.AlternatingLeastSquares(
        factors=factors, iterations=iterations,
        regularization=regularization, alpha=alpha, use_gpu=False)
    t0 = time.time()
    model.fit(user_item.tocsr())
    print(f'  ALS fitted {time.time()-t0:.1f}s | factors={factors} alpha={alpha}')
    return model, user_item

def als_recommend(model, user_item, user_ids, k=10):
    recs = {}
    for uid in user_ids:
        ids, _ = model.recommend(uid, user_item[uid], N=k,
                                 filter_already_liked_items=True)
        recs[uid] = ids.tolist()
    return recs

print('ALS functions defined.')

ALS functions defined.


In [9]:
# Tune ALS on validation set
val_users_lastfm = lastfm_val.user_id.unique()
gt_val_lastfm    = lastfm_val.groupby('user_id')['item_id'].apply(list).to_dict()
als_results      = []

for factors, alpha in tqdm(list(itertools.product([64,128,256],[1.0,10.0,40.0])),
                            desc='ALS tuning'):
    model, ui = fit_als(lastfm_train, n_users_lastfm, n_items_lastfm,
                        factors=factors, alpha=alpha)
    recs = als_recommend(model, ui, val_users_lastfm, k=10)
    als_results.append({'factors':factors,'alpha':alpha,
                        **metrics(recs, gt_val_lastfm)})

als_df = pd.DataFrame(als_results).sort_values('ndcg@10', ascending=False)
print('ALS tuning results:')
als_df

ALS tuning:   0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  ALS fitted 0.5s | factors=64 alpha=1.0


  0%|          | 0/20 [00:00<?, ?it/s]

  ALS fitted 0.5s | factors=64 alpha=10.0


...

  0%|          | 0/20 [00:00<?, ?it/s]

  ALS fitted 4.0s | factors=256 alpha=10.0


  0%|          | 0/20 [00:00<?, ?it/s]

  ALS fitted 3.8s | factors=256 alpha=40.0
ALS tuning results:


,factors,alpha,hr@10,ndcg@10,map@10
1,64,10.0,0.602695,0.194680,0.111053
2,64,40.0,0.588140,0.181455,0.101123
4,128,10.0,0.542857,0.161444,0.089938
5,128,40.0,0.539623,0.157595,0.086593
0,64,1.0,0.477628,0.130206,0.068975
8,256,40.0,0.442588,0.121956,0.066346
7,256,10.0,0.424259,0.119686,0.066027
3,128,1.0,0.414016,0.112436,0.060320
6,256,1.0,0.362803,0.100669,0.055075


In [10]:
# Best ALS on test set
best_als_row = als_df.iloc[0]
best_als_model, best_ui = fit_als(
    lastfm_train, n_users_lastfm, n_items_lastfm,
    factors=int(best_als_row.factors), alpha=float(best_als_row.alpha))
als_recs_lastfm = als_recommend(best_als_model, best_ui, lastfm_test.user_id.unique(), k=10)
als_m_lastfm    = metrics(als_recs_lastfm, gt_lastfm)
print(f'Best ALS (factors={int(best_als_row.factors)}, alpha={best_als_row.alpha}) test metrics:')
print(als_m_lastfm)

print('\nCold-start breakdown:')
for group, m in cold_start_metrics(als_recs_lastfm, lastfm_test).items():
    print(f'  {group}: {m}')

  0%|          | 0/20 [00:00<?, ?it/s]

  ALS fitted 0.4s | factors=64 alpha=10.0
Best ALS (factors=64, alpha=10.0) test metrics:
{'hr@10': 0.7196765498652291, 'ndcg@10': 0.23979504393365836, 'map@10': 0.13874936892996192}

Cold-start breakdown:
  cold_start: {'hr@10': 0.2, 'ndcg@10': 0.2, 'map@10': 0.2}
  regular: {'hr@10': 0.721081081081081, 'ndcg@10': 0.23990259810645204, 'map@10': 0.13858382668382668}


## Results summary

The results summary synthesises the empirical performance of three baselines across two datasets with distinct interaction characteristics. MovieLens 1M is treated as an explicit-feedback dataset using `surprise.SVD`, while LastFM-2K is modelled as implicit feedback with `implicit.ALS` after constructing a user-item frequency matrix. The evaluation emphasises ranking quality through `hr@10`, `ndcg@10`, and `map@10`, and the comparison highlights how a popularity baseline provides a robust, low-variance benchmark against which matrix factorization methods can be assessed. This structure supports a clear methodological contrast: explicit factorization is calibrated on MovieLens validation data, and implicit ALS is tuned for LastFM user behavior, enabling a direct comparison of practical recommendation performance and cold-start resilience.

In [11]:
summary = pd.DataFrame([
    {'Dataset':'MovieLens 1M', 'Model':'Popularity', **pop_m_ml},
    {'Dataset':'MovieLens 1M', 'Model':'SVD (best)', **svd_m_ml},
    {'Dataset':'LastFM-2K',   'Model':'Popularity', **pop_m_lastfm},
    {'Dataset':'LastFM-2K',   'Model':'ALS (best)', **als_m_lastfm},
]).round(4)

summary.to_csv(PROC / 'baseline_results.csv', index=False)
print('Results saved to data/processed/baseline_results.csv')
summary

Results saved to data/processed/baseline_results.csv


,Dataset,Model,hr@10,ndcg@10,map@10
0,MovieLens 1M,Popularity,0.2505,0.0533,0.0258
1,MovieLens 1M,SVD (best),0.0535,0.0055,0.0015
2,LastFM-2K,Popularity,0.3423,0.1024,0.0568
3,LastFM-2K,ALS (best),0.7197,0.2398,0.1387


In [13]:
# Save best model configs for use in deep learning notebooks
baseline_config = {
    'best_svd': {'n_factors': int(svd_df.iloc[0].n_factors),
                 'reg_all':   float(svd_df.iloc[0].reg_all)},
    'best_als': {'factors': int(als_df.iloc[0].factors),
                 'alpha':   float(als_df.iloc[0].alpha)},
    'baseline_metrics': {
        'movielens_popularity': pop_m_ml,
        'movielens_svd':        svd_m_ml,
        'lastfm_popularity':    pop_m_lastfm,
        'lastfm_als':           als_m_lastfm,
    }
}
with open(PROC / 'baseline_config.pkl', 'wb') as f:
    pickle.dump(baseline_config, f)
print('Baseline config saved. Run 03_ncf.ipynb next.')

Baseline config saved. Run 03_ncf.ipynb next.
